# Train test split

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
df = pd.read_csv("../data/processed/processed_insurance.csv")
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


In [8]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="charges")
y = df.charges
np.random.seed(42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Lets try PolynomialFeatures

In [9]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

poly = PolynomialFeatures(degree=2)
scaler = StandardScaler()

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

In [10]:
X_train_scaled.shape, X_train.shape

((1069, 45), (1069, 8))

In [13]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

baseline_model = LinearRegression()
ridge_model = Ridge(alpha=10.0)
lasso_model = Lasso(alpha=100.0, max_iter=10000, random_state=42)
elastic_model = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000, random_state=42)

baseline_model.fit(X_train_scaled, y_train)
ridge_model.fit(X_train_scaled, y_train)
lasso_model.fit(X_train_scaled, y_train)
elastic_model.fit(X_train_scaled, y_train)

,alpha,1.0
,l1_ratio,0.5
,fit_intercept,True
,precompute,False
,max_iter,10000
,copy_X,True
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


In [14]:
baseline_pred = baseline_model.predict(X_test_scaled)
ridge_pred = ridge_model.predict(X_test_scaled)
lasso_pred = lasso_model.predict(X_test_scaled)
elastic_pred = elastic_model.predict(X_test_scaled)

In [19]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error, mean_absolute_error

model_preds = {"LinearRegression": baseline_pred,
               "Ridge": ridge_pred, 
               "Lasso": lasso_pred, 
               "Elastic": elastic_pred}

for model_name, pred in model_preds.items():
    r2 = r2_score(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    
    print(f"Model: {model_name:18} | R2: {r2:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")

Model: LinearRegression   | R2: 0.8825 | RMSE: 4646.06 | MAE: 2867.32
Model: Ridge              | R2: 0.8800 | RMSE: 4696.26 | MAE: 2933.53
Model: Lasso              | R2: 0.8782 | RMSE: 4731.73 | MAE: 2948.97
Model: Elastic            | R2: 0.8084 | RMSE: 5933.41 | MAE: 4121.96
